In [1]:
"""
First spin up vLLM server with:

vllm serve Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4 \
  --quantization gptq_marlin \
  --dtype auto \
  --max-model-len 4096 \
  --gpu-memory-utilization 0.85 \
  --max-num-seqs 16 \
  --tensor-parallel-size 2 --data-parallel-size 1 --enable-expert-parallel --enable-eplb
  
Can also profile with:

vllm bench serve \
  --backend vllm \
  --model Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4 \
  --dataset-name random \
  --num-prompts 50 \
  --request-rate 10 \
  --result-filename metrics.json
  
"""

'\nFirst spin up vLLM server with:\n\nvllm serve Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4   --quantization gptq_marlin   --dtype auto   --max-model-len 4096   --gpu-memory-utilization 0.85   --max-num-seqs 16   --tensor-parallel-size 2 --data-parallel-size 1 --enable-expert-parallel --enable-eplb\n  \nCan also profile with:\n\nvllm bench serve   --backend vllm   --model Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4   --dataset-name random   --num-prompts 50   --request-rate 10   --result-filename metrics.json\n  \n'

In [2]:
import os
# Force vLLM to use spawn method to avoid CUDA fork errors in Jupyter
#os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# Tell the C++ compiler's linker exactly where to find Conda's CUDA libraries
#conda_prefix = os.environ.get("CONDA_PREFIX", "/home/dylan/miniconda3")
#os.environ["LIBRARY_PATH"] = f"{conda_prefix}/lib:" + os.environ.get("LIBRARY_PATH", "")

In [3]:
from experiments_vllm import *
from data import *
from transformers import AutoTokenizer

In [4]:
# Set seeds
seed = 43
torch.manual_seed(seed);

In [5]:
# Configuration
# For full dataset: n_samples = 15000, max_new_tokens = 100, batch_size = 16
model = "Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4"
port = 8000
n_samples = 100
max_new_tokens = 100
max_model_len = 1024
gpu_memory_utilization = 0.85
n_gpus = 1
batch_size = 16
n_warmup_samples = 2

In [6]:
# Select prompts
sample_prompts = [
    "Explain the theory of relativity in simple terms.",
    "Write a python script to scrape a website.",
    "What are the benefits of MoE (Mixture of Experts) architectures?",
    "Tell me a short sci-fi story about a sentient coffee machine.",
    "Summarize the history of the Roman Empire in 3 paragraphs."
]

In [7]:
# Main experiment
results = await run_experiment_vllm_throughput(model,
                                               sample_prompts,
                                               seed=seed,
                                               max_new_tokens=max_new_tokens,
                                               max_model_len=max_model_len,
                                               gpu_memory_utilization=gpu_memory_utilization,
                                               n_gpus=n_gpus,
                                               print_output=True)

Starting vLLM server for Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4...
Waiting for server to initialize ...


/home/dylan/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


(APIServer pid=907482) INFO 04-27 11:32:43 [utils.py:299] 
(APIServer pid=907482) INFO 04-27 11:32:43 [utils.py:299]        █     █     █▄   ▄█
(APIServer pid=907482) INFO 04-27 11:32:43 [utils.py:299]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.19.1
(APIServer pid=907482) INFO 04-27 11:32:43 [utils.py:299]   █▄█▀ █     █     █     █  model   Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4
(APIServer pid=907482) INFO 04-27 11:32:43 [utils.py:299]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=907482) INFO 04-27 11:32:43 [utils.py:299] 
(APIServer pid=907482) INFO 04-27 11:32:43 [utils.py:233] non-default args: {'model_tag': 'Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4', 'model': 'Qwen/Qwen1.5-MoE-A2.7B-Chat-GPTQ-Int4', 'seed': 43, 'max_model_len': 1024, 'quantization': 'gptq_marlin', 'enable_expert_parallel': True, 'gpu_memory_utilization': 0.85, 'max_num_seqs': 16}


(APIServer pid=907482) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


(APIServer pid=907482) INFO 04-27 11:32:44 [model.py:549] Resolved architecture: Qwen2MoeForCausalLM
(APIServer pid=907482) INFO 04-27 11:32:44 [model.py:1678] Using max model len 1024
(APIServer pid=907482) INFO 04-27 11:32:44 [gptq_marlin.py:229] The model is convertible to gptq_marlin during runtime. Using gptq_marlin kernel.
(APIServer pid=907482) INFO 04-27 11:32:44 [vllm.py:790] Asynchronous scheduling is enabled.


...

(EngineCore pid=907543) INFO 04-27 11:33:16 [vllm.py:790] Asynchronous scheduling is enabled.
(APIServer pid=907482) INFO 04-27 11:33:16 [api_server.py:592] Supported tasks: ['generate']
(APIServer pid=907482) WARNING 04-27 11:33:16 [model.py:1435] Default vLLM sampling parameters have been overridden by the model's `generation_config.json`: `{'repetition_penalty': 1.05, 'temperature': 0.7, 'top_k': 20, 'top_p': 0.8}`. If this is not intended, please relaunch vLLM instance with `--generation-config vllm`.
(APIServer pid=907482) INFO 04-27 11:33:18 [hf.py:314] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
(APIServer pid=907482) INFO 04-27 11:33:19 [api_server.py:596] Starting vLLM server on http://0.0.0.0:8000
(APIServer pid=907482) INFO 04-27 11:33:19 [launcher.py:37] Available routes are:
(APIServer pid=907482) INFO 04-27 11:33:19 [launcher.py:46] Route: /openapi.json, Methods: HEAD, GET
(APIServer pid=907482) 

(APIServer pid=907482) INFO:     Started server process [907482]
(APIServer pid=907482) INFO:     Waiting for application startup.
(APIServer pid=907482) INFO:     Application startup complete.


(APIServer pid=907482) INFO:     127.0.0.1:59712 - "GET /v1/models HTTP/1.1" 200 OK
Server is ready!
Sending batch of 5 concurrent requests...
(APIServer pid=907482) INFO:     127.0.0.1:59722 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=907482) INFO:     127.0.0.1:59738 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=907482) INFO:     127.0.0.1:59750 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=907482) INFO:     127.0.0.1:59766 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=907482) INFO:     127.0.0.1:59776 - "POST /v1/chat/completions HTTP/1.1" 200 OK

--- Per-Request Metrics ---
Request 0: TTFT = 0.7275s | TPOT = 12.11ms | Tokens = 100
Request 1: TTFT = 0.2398s | TPOT = 12.14ms | Tokens = 100
Request 2: TTFT = 0.2371s | TPOT = 12.15ms | Tokens = 100
Request 3: TTFT = 0.2337s | TPOT = 12.10ms | Tokens = 100
Request 4: TTFT = 0.2331s | TPOT = 12.15ms | Tokens = 100

--- Batch Metrics ---
Average Per-Request TPOT: 12.13 ms/token

(APIServer pid=907482) INFO:     Shutting down
[rank0]:[W427 11:33:26.215858765 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())
(APIServer pid=907482) INFO:     Waiting for application shutdown.
(APIServer pid=907482) INFO:     Application shutdown complete.
(APIServer pid=907482) INFO:     Finished server process [907482]


Server successfully shut down.


In [8]:
print(results)

[{'prompt': 'Explain the theory of relativity in simple terms.', 'prompt_id': 0, 'ttft': 0.7275056261569262, 'tpot': 0.012109277431260456, 'num_output_tokens': 100, 'total_time': 1.9263240918517113, 'response': '\n\nThe theory of relativity is a set of concepts in physics that describe how the laws of physics work in different frames of reference, or when viewed from different perspectives. \n\nThere are two main parts to the theory: special relativity and general relativity.\n\nSpecial relativity was developed by Albert Einstein in the early 1900s and it introduced the concept of time dilation and length contraction. According to special relativity, the laws of physics are the same for all observers in uniform motion'}, {'prompt': 'Write a python script to scrape a website.', 'prompt_id': 1, 'ttft': 0.2397836558520794, 'tpot': 0.012138085289284437, 'num_output_tokens': 100, 'total_time': 1.4414540994912386, 'response': "Sure, here's an example Python script that uses the `requests` an